# Week 4: Data Quality Checks (TalentRank)

Small additional checks supporting `docs/week4_data_quality_report.md`.

- **Audited output:** Topic 3 Part C `data/pipeline_v1/tabular/prepared_table.csv` + `pipeline_manifest.json`
- **Before-cleaning comparison:** `data/raw/employee_promotion.csv`
- **Topic 2 requirements audited:** FR3 / NFR1 (valid target and inputs for prediction) and RAI1 (fair top-of-list selection)

In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd

PROJECT_DIR = Path.cwd().parent
AUDIT_PATH = PROJECT_DIR / "data/pipeline_v1/tabular/prepared_table.csv"
MANIFEST_PATH = PROJECT_DIR / "data/pipeline_v1/tabular/pipeline_manifest.json"
RAW_PATH = PROJECT_DIR / "data/raw/employee_promotion.csv"

sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
assert manifest["validation"]["output_sha256"][0] == sha(AUDIT_PATH), "Audited file does not match its manifest"

before = pd.read_csv(RAW_PATH)      # before cleaning (raw)
df = pd.read_csv(AUDIT_PATH)        # after cleaning (audited output)
print("Manifest hash verified:", sha(AUDIT_PATH)[:12], "...")
print("Before cleaning:", before.shape, "| After cleaning:", df.shape)

Manifest hash verified: 1dd337e5f887 ...
Before cleaning: (54808, 13) | After cleaning: (54808, 13)


## 1. Overall promotion outcome (target imbalance)

In [2]:
t = df["is_promoted"].value_counts().rename({0: "Not promoted (0)", 1: "Promoted (1)"})
overall = pd.DataFrame({"Employees": t, "Percent": (t / len(df) * 100).round(2)})
overall.loc["Total"] = [len(df), 100.0]
overall

,Employees,Percent
is_promoted,,
Not promoted (0),50140.0,91.48
Promoted (1),4668.0,8.52
Total,54808.0,100.00


## 2. Department overview (all nine departments)

In [3]:
dept = df.groupby("department")["is_promoted"].agg(Total="size", Promoted="sum")
dept["Not promoted"] = dept["Total"] - dept["Promoted"]
dept["Promoted (%)"] = (dept["Promoted"] / dept["Total"] * 100).round(2)
dept["Not promoted (%)"] = (dept["Not promoted"] / dept["Total"] * 100).round(2)
dept["Share of all employees (%)"] = (dept["Total"] / len(df) * 100).round(2)
dept = dept.sort_values("Total", ascending=False)
dept.loc["All departments"] = [len(df), df["is_promoted"].sum(), (df["is_promoted"] == 0).sum(),
                               round(df["is_promoted"].mean() * 100, 2),
                               round((1 - df["is_promoted"].mean()) * 100, 2), 100.0]
dept

,Total,Promoted,Not promoted,Promoted (%),Not promoted (%),Share of all employees (%)
department,,,,,,
Sales & Marketing,16840.0,1213.0,15627.0,7.20,92.80,30.73
Operations,11348.0,1023.0,10325.0,9.01,90.99,20.71
Procurement,7138.0,688.0,6450.0,9.64,90.36,13.02
Technology,7138.0,768.0,6370.0,10.76,89.24,13.02
Analytics,5352.0,512.0,4840.0,9.57,90.43,9.76
Finance,2536.0,206.0,2330.0,8.12,91.88,4.63
HR,2418.0,136.0,2282.0,5.62,94.38,4.41
Legal,1039.0,53.0,986.0,5.10,94.90,1.90
R&D,999.0,69.0,930.0,6.91,93.09,1.82


## 3. Two selected departments: Technology (highest promotion rate) and Legal (lowest promotion rate, small department)

In [4]:
sel = df[df["department"].isin(["Technology", "Legal"])]
detail = sel.groupby("department").agg(
    Total=("is_promoted", "size"),
    Promoted=("is_promoted", "sum"),
    Missing_education=("education", lambda s: s.isna().sum()),
    Missing_rating=("previous_year_rating", lambda s: s.isna().sum()),
    Missing_training_score=("avg_training_score", lambda s: s.isna().sum()),
    Mean_training_score=("avg_training_score", "mean"),
    Female_share_pct=("gender", lambda s: (s == "f").mean() * 100),
)
detail["Not promoted"] = detail["Total"] - detail["Promoted"]
detail["Promoted (%)"] = detail["Promoted"] / detail["Total"] * 100
for c in ["Missing_education", "Missing_rating", "Missing_training_score"]:
    detail[c + " (%)"] = detail[c] / detail["Total"] * 100
detail.round(2).T

department,Legal,Technology
Total,1039.00,7138.00
Promoted,53.00,768.00
Missing_education,4.00,99.00
Missing_rating,168.00,546.00
Missing_training_score,123.00,0.00
Mean_training_score,59.72,79.93
Female_share_pct,14.34,39.06
Not promoted,986.00,6370.00
Promoted (%),5.10,10.76
Missing_education (%),0.38,1.39


## 4. Completeness: before vs after cleaning

In [5]:
comp = pd.DataFrame({
    "Missing before cleaning": before.isna().sum(),
    "Missing after cleaning": df.isna().sum(),
})
comp["Percent after"] = (comp["Missing after cleaning"] / len(df) * 100).round(2)
print(comp[comp["Missing before cleaning"] > 0])
print("\nis_promoted missing:", int(df["is_promoted"].isna().sum()))

print("\nWhy is previous_year_rating missing? Length of service of records with a missing rating:")
print(df.loc[df["previous_year_rating"].isna(), "length_of_service"].value_counts().to_string())
one_year = df[df["length_of_service"] == 1]
print(f"\nEmployees with 1 year of service: {len(one_year)}; with missing rating: "
      f"{one_year['previous_year_rating'].isna().sum()} ({one_year['previous_year_rating'].isna().mean():.1%})")

print("\nMissing avg_training_score by department (%):")
print((df.groupby("department")["avg_training_score"].apply(lambda s: s.isna().mean() * 100)).round(1).sort_values(ascending=False).to_string())
print("\nPromotion rate when education is missing vs present (%):")
print((df.groupby(df["education"].isna())["is_promoted"].mean() * 100).round(2).rename({False: "present", True: "missing"}).to_string())

                      Missing before cleaning  Missing after cleaning  \
education                                2409                    2409   
previous_year_rating                     4124                    4124   
avg_training_score                       2560                    2560   

                      Percent after  
education                      4.40  
previous_year_rating           7.52  
avg_training_score             4.67  

is_promoted missing: 0

Why is previous_year_rating missing? Length of service of records with a missing rating:
length_of_service
1    4124

Employees with 1 year of service: 4547; with missing rating: 4124 (90.7%)

Missing avg_training_score by department (%):
department
Finance              12.4
Operations           12.3
Legal                11.8
Sales & Marketing     3.8
HR                    3.7
Analytics             0.0
Procurement           0.0
R&D                   0.0
Technology            0.0

Promotion rate when education is missing vs p

## 5. Audit sample (20 records) and accuracy spot-check

14 random records stratified by outcome (7 promoted, 7 not; seed 42) + 6 known problem cases (2 each with missing education, rating and training score). Employee IDs are not displayed.

In [6]:
SEED = 42
sample = pd.concat([
    df[df["is_promoted"] == 1].sample(7, random_state=SEED).assign(selection="random"),
    df[df["is_promoted"] == 0].sample(7, random_state=SEED).assign(selection="random"),
    df[df["education"].isna()].sample(2, random_state=SEED).assign(selection="missing education"),
    df[df["previous_year_rating"].isna()].sample(2, random_state=SEED).assign(selection="missing rating"),
    df[df["avg_training_score"].isna()].sample(2, random_state=SEED).assign(selection="missing training score"),
])
sample.insert(0, "item_id", [f"S{i:02d}" for i in range(1, 21)])
print(sample.drop(columns="employee_id").to_string(index=False))

raw_idx = before.set_index("employee_id")
cols = [c for c in before.columns if c != "employee_id"]
match = sum(all((pd.isna(r[c]) and pd.isna(raw_idx.loc[r["employee_id"], c])) or r[c] == raw_idx.loc[r["employee_id"], c]
                for c in cols) for _, r in sample.iterrows())
print(f"\nSampled prepared records identical to the raw source: {match} / {len(sample)}")

item_id        department    region        education gender recruitment_channel  no_of_trainings  age  previous_year_rating  length_of_service  awards_won  avg_training_score  is_promoted              selection
    S01         Analytics region_31 Master's & above      m               other                1   35                   3.0                  8           0                84.0            1                 random
    S02 Sales & Marketing region_28       Bachelor's      f               other                1   29                   5.0                  2           1                64.0            1                 random
    S03       Procurement  region_2       Bachelor's      f            sourcing                1   42                   4.0                  2           1                80.0            1                 random
    S04        Operations region_19 Master's & above      m               other                1   37                   5.0                  4           1  

## 6. Validation rules

- **V1 (FR3 / NFR1):** `is_promoted` present and in {0, 1}.
- **V2 (FR2 / NFR1):** `previous_year_rating` in {1..5} or missing only when `length_of_service` = 1; `avg_training_score` in [0, 100]; `length_of_service` ≤ `age` − 18.

In [7]:
v1_fail = df[~df["is_promoted"].isin([0, 1])]
rating_ok = df["previous_year_rating"].isin([1, 2, 3, 4, 5]) | (df["previous_year_rating"].isna() & (df["length_of_service"] == 1))
score_ok = df["avg_training_score"].between(0, 100) | df["avg_training_score"].isna()
service_ok = df["length_of_service"] <= df["age"] - 18
v2_fail = df[~(rating_ok & score_ok & service_ok)]
print(f"V1 tested {len(df)}: pass {len(df) - len(v1_fail)}, fail {len(v1_fail)}")
print(f"V2 tested {len(df)}: pass {len(df) - len(v2_fail)}, fail {len(v2_fail)} "
      f"(rating {int((~rating_ok).sum())}, score {int((~score_ok).sum())}, service-vs-age {int((~service_ok).sum())})")
print(v2_fail[["department", "age", "length_of_service", "previous_year_rating", "is_promoted"]].to_string())
print("\nValue ranges (full data):")
print(df[["no_of_trainings", "age", "previous_year_rating", "length_of_service", "avg_training_score"]].agg(["min", "max"]).T)

V1 tested 54808: pass 54808, fail 0
V2 tested 54808: pass 54807, fail 1 (rating 0, score 0, service-vs-age 1)
       department  age  length_of_service  previous_year_rating  is_promoted
35422  Operations   30                 13                   4.0            0

Value ranges (full data):
                       min   max
no_of_trainings        1.0  10.0
age                   20.0  60.0
previous_year_rating   1.0   5.0
length_of_service      1.0  37.0
avg_training_score    39.0  99.0


## 7. Coverage and bias signals (gender, age, training-score scale)

In [8]:
g = df.groupby("gender")["is_promoted"].agg(Employees="size", Promoted="sum", Rate="mean")
g["Rate"] = (g["Rate"] * 100).round(2)
print(g, "\nGender ratio (lower/higher rate):", round(g["Rate"].min() / g["Rate"].max(), 3))

bands = pd.cut(df["age"], [19, 29, 39, 49, 60], labels=["20-29", "30-39", "40-49", "50-60"])
a = df.groupby(bands, observed=True)["is_promoted"].agg(Employees="size", Promoted="sum", Rate="mean")
a["Rate"] = (a["Rate"] * 100).round(2)
print("\n", a, "\nAge ratio (lowest/highest band):", round(a["Rate"].min() / a["Rate"].max(), 3))

print("\nMean avg_training_score by department:")
print(df.groupby("department")["avg_training_score"].mean().round(1).sort_values().to_string())

        Employees  Promoted  Rate
gender                           
f           16312      1467  8.99
m           38496      3201  8.32 
Gender ratio (lower/higher rate): 0.925

        Employees  Promoted  Rate
age                             
20-29      14453      1226  8.48
30-39      28030      2556  9.12
40-49       8952       664  7.42
50-60       3373       222  6.58 
Age ratio (lowest/highest band): 0.721

Mean avg_training_score by department:
department
HR                   50.2
Sales & Marketing    50.4
Legal                59.7
Finance              60.1
Operations           60.1
Procurement          70.1
Technology           79.9
Analytics            84.6
R&D                  84.6
